# Phase 09 — Evaluation, Observability & Docker

This notebook extends the Phase 09 evaluation framework to cover the full multimodal RAG pipeline.

## Sections
- **9.1** Evaluation dataset
- **9.2** Text retrieval metrics (baseline)
- **9.3** Figure Recall@k
- **9.4** Multimodal Recall@k
- **9.5** Answer faithfulness
- **9.6** Citation correctness
- **9.7** Compare four RAG variants
- **9.8** Observability — OpenTelemetry span attributes
- **9.9** Docker / reproducibility notes

## Learning Objectives
- Understand the difference between text-only and multimodal retrieval metrics
- Run `figure_recall_at_k` and `multimodal_recall_at_k` on a benchmark dataset
- Evaluate citation correctness for `[T#]`/`[V#]` labelled answers
- Instrument a multimodal RAG call with OpenTelemetry spans
- Know what Docker setup is needed for multimodal mode vs text-only mode

## 9.1 Evaluation Dataset

A small multimodal benchmark is stored in `tests/evaluation/multimodal_benchmark.json`.

Each item has:
- `question` — the query to answer
- `question_type` — `text_only`, `figure_lookup`, `visual_reasoning`, `mixed_text_visual`
- `expected_answer` — reference answer for human review
- `relevant_text_pages` — page numbers that should appear in text evidence
- `relevant_figures` — `{source, page, figure_index}` dicts
- `relevant_modalities` — which modalities a good system needs

The dataset is intentionally small (8–10 items). For a production evaluation you
would annotate a larger set from your actual document corpus.

In [ ]:
import json
from pathlib import Path

benchmark_path = Path("tests/evaluation/multimodal_benchmark.json")
benchmark = json.loads(benchmark_path.read_text())

print(f"Total questions: {len(benchmark)}")
for item in benchmark:
    print(f"  [{item['question_type']:20s}] {item['question'][:70]}")

In [ ]:
from collections import Counter
type_counts = Counter(item["question_type"] for item in benchmark)
print("Question type distribution:")
for qtype, count in sorted(type_counts.items()):
    print(f"  {qtype:25s}: {count}")

## 9.2 Text Retrieval Metrics (Baseline)

Before evaluating the multimodal pipeline, establish a text-only baseline
using the existing `recall_at_k`, `mrr`, and `ndcg_at_k` metrics.

In [ ]:
from mrta.evaluation.metrics import recall_at_k, mrr, ndcg_at_k, citation_coverage

# Simulate a retrieval result for a text question
retrieved_sources = ["sample.pdf", "attention.pdf", "sample.pdf"]
expected_docs = ["sample.pdf"]

print(f"Recall@1:  {recall_at_k(retrieved_sources, expected_docs, k=1):.3f}")
print(f"Recall@3:  {recall_at_k(retrieved_sources, expected_docs, k=3):.3f}")
print(f"MRR:       {mrr(retrieved_sources, expected_docs):.3f}")
print(f"nDCG@3:    {ndcg_at_k(retrieved_sources, expected_docs, k=3):.3f}")
print(f"Coverage:  {citation_coverage(retrieved_sources, expected_docs):.3f}")

## 9.3 Figure Recall@k

`figure_recall_at_k` measures whether the system retrieved the relevant figures.

A figure is identified by `(source, page, figure_index)`. The metric is:

```
number of relevant figures in top-k visual results
─────────────────────────────────────────────────
total number of relevant figures
```

We evaluate at k=1, 3, 5.

In [ ]:
from mrta.core.schemas import EvidenceRecord
from mrta.evaluation.multimodal_metrics import figure_recall_at_k

# Construct synthetic retrieved records to demonstrate the metric
def make_image_record(page: int, figure_index: int, source: str = "sample.pdf") -> EvidenceRecord:
    return EvidenceRecord(
        evidence_id=f"{source}_p{page}_f{figure_index}",
        doc_id="doc1",
        source=source,
        page=page,
        modality="image",
        figure_index=figure_index,
    )

retrieved = [
    make_image_record(2, 1),  # rank 1 — correct
    make_image_record(3, 2),  # rank 2 — not expected
    make_image_record(1, 1),  # rank 3 — not expected
]
expected_figures = [{"source": "sample.pdf", "page": 2, "figure_index": 1}]

for k in [1, 3, 5]:
    score = figure_recall_at_k(retrieved, expected_figures, k=k)
    print(f"Figure Recall@{k}: {score:.3f}")

## 9.4 Multimodal Recall@k

`multimodal_recall_at_k` reports text and visual recall **separately** and
combines them as a geometric mean.

Why geometric mean? If text recall is 1.0 but visual recall is 0.0, the system
failed for mixed questions — the overall score should reflect that. A simple
average would give 0.5, hiding the failure.

In [ ]:
import math
from mrta.core.schemas import EvidenceRecord
from mrta.evaluation.multimodal_metrics import multimodal_recall_at_k

def make_text_record(page: int, source: str = "sample.pdf") -> EvidenceRecord:
    return EvidenceRecord(
        evidence_id=f"{source}_p{page}_text",
        doc_id="doc1",
        source=source,
        page=page,
        modality="text",
        text="Some text.",
    )

# Mixed scenario: correct text page retrieved, correct figure also retrieved
retrieved_mixed = [
    make_text_record(2),
    make_image_record(2, 1),
]
result = multimodal_recall_at_k(
    retrieved_mixed,
    expected_text_pages=[2],
    expected_figures=[{"source": "sample.pdf", "page": 2, "figure_index": 1}],
    k=5,
)
print("Perfect retrieval (text + visual):")
for k, v in result.items():
    print(f"  {k:10s}: {v:.3f}")

print()

# Failure scenario: text retrieved but figure missed
result_text_only = multimodal_recall_at_k(
    [make_text_record(2)],
    expected_text_pages=[2],
    expected_figures=[{"source": "sample.pdf", "page": 2, "figure_index": 1}],
    k=5,
)
print("Text only retrieved (visual missed):")
for k, v in result_text_only.items():
    print(f"  {k:10s}: {v:.3f}")

## 9.5 Answer Faithfulness

The existing `faithfulness` metric checks whether answer sentences are grounded
in retrieved text chunks.

**Limitation for multimodal answers**: a text-only judge cannot independently
verify raw visual semantics. When an answer cites a figure (`[V1]`) and the
figure has no textual description in the retrieved evidence, faithfulness will
be pessimistically low.

Mitigation: ensure visual evidence includes VLM-generated captions or
`detailed_description` so the faithfulness checker has something to compare.

In [ ]:
from mrta.core.schemas import Chunk
from mrta.evaluation.metrics import faithfulness

chunk = Chunk(
    chunk_id="c1",
    doc_id="doc1",
    source="sample.pdf",
    page=2,
    text="Attention mechanisms have become integral to sequence modelling.",
)

well_grounded = "Attention mechanisms are integral to sequence modelling [T1]."
hallucinated  = "Transformers use quantum computing for embeddings [V1]."

print(f"Faithfulness (grounded):     {faithfulness(well_grounded, [chunk]):.3f}")
print(f"Faithfulness (hallucinated): {faithfulness(hallucinated,  [chunk]):.3f}")

print()
print("Note: visual-only answers may score low because the text judge")
print("cannot verify image content. This is expected behaviour — document it.")

## 9.6 Citation Correctness

`multimodal_citation_correctness` evaluates three levels of citation quality:

| Score | Meaning |
|---|---|
| `format` | Citation labels are correctly written (`[T1]`, `[V2]`, not `[t1]` or `[T 1]`) |
| `provenance` | Every cited label maps to an actual citation in the answer |
| `support` | Cited source/page is in the retrieved evidence list |

In [ ]:
from mrta.core.schemas import MultimodalAnswer, MultimodalCitation
from mrta.evaluation.multimodal_metrics import multimodal_citation_correctness

answer = MultimodalAnswer(
    answer="Attention is described in [T1]. Figure 1 [V1] shows the architecture.",
    text_citations=[
        MultimodalCitation(
            label="[T1]", evidence_id="t1", modality="text",
            source="sample.pdf", page=2,
        )
    ],
    visual_citations=[
        MultimodalCitation(
            label="[V1]", evidence_id="v1", modality="image",
            source="sample.pdf", page=2, figure_index=1,
        )
    ],
    retrieval_mode="multimodal",
    latency_s=1.2,
)

retrieved_ev = [
    make_text_record(2),
    make_image_record(2, 1),
]

result = multimodal_citation_correctness(answer, retrieved_ev)
print("Citation correctness:")
for k, v in result.items():
    print(f"  {k:12s}: {v:.3f}")

## 9.7 Compare Four RAG Variants

The table below shows the architectural differences between the four systems
we have built. Run the eval pipeline (`run_multimodal_eval`) against each
when a real indexed corpus is available.

| System | Retrieval | Generation | Figure Recall |
|---|---|---|---|
| Text RAG | VectorStore only | LLMClient.chat() | Not applicable |
| Caption RAG | CaptionVectorStore | LLMClient.chat() | Via caption similarity |
| CLIP RAG | VisualVectorStore | LLMClient.chat() | Via CLIP text→image |
| Full Multimodal RAG | MultimodalRetriever (RRF fusion) | VLMClient.generate() + images | Direct figure retrieval |

Expected relative performance:

| Question type | Best system |
|---|---|
| text_only | Text RAG (fast, no VLM overhead) |
| figure_lookup | CLIP RAG or Full Multimodal |
| visual_reasoning | Full Multimodal RAG |
| mixed_text_visual | Full Multimodal RAG |

In [ ]:
from mrta.evaluation.multimodal_eval_pipeline import run_multimodal_eval

# To run a real comparison, initialise each RAG variant with a loaded corpus:
#
#   from mrta import MultimodalRAG, MultimodalRetriever
#   rag = MultimodalRAG(retriever=retriever, vlm=vlm)
#   report = run_multimodal_eval(benchmark, rag)
#
# The report exposes per-type breakdowns:
#   report.by_type["text_only"]["overall_recall"]
#   report.by_type["figure_lookup"]["figure_recall_at_5"]

print("MultimodalEvalReport fields:")
from mrta.evaluation.multimodal_eval_pipeline import MultimodalEvalReport
import dataclasses
for f in dataclasses.fields(MultimodalEvalReport):
    print(f"  {f.name}")

## 9.8 Observability — OpenTelemetry Span Attributes

Every `MultimodalRAG.ask()` call emits a `mrta.multimodal_rag.ask` span and
every `MultimodalRetriever.retrieve_with_fusion_details()` call emits a
`mrta.multimodal_retriever.retrieve` span.

### Retrieval span attributes
```
retrieval.text_candidates       — number of text records retrieved
retrieval.visual_candidates     — number of CLIP visual records retrieved
retrieval.final_candidates      — number of records after RRF fusion
retrieval.text_top_score        — cosine similarity of top text result
retrieval.visual_top_score      — cosine similarity of top visual result
retrieval.fusion_method         — always "rrf"
retrieval.rrf_k                 — RRF smoothing constant
latency.text_retrieval          — seconds for text search
latency.visual_retrieval        — seconds for CLIP search
```

### Generation span attributes
```
generation.text_evidence_count  — text records passed to VLM
generation.visual_evidence_count— visual records passed to VLM
generation.vision_model         — model name from VLMClient
generation.teaching_mode        — teaching mode or "none"
generation.retrieval_mode       — "multimodal" or "text_only" (fallback)
latency.vlm                     — VLM generation time in seconds
```

In [ ]:
# Enable console tracing to see spans printed to stdout
from mrta.observability.tracing import configure_tracer
configure_tracer(service_name="mrta-notebook", console=True)

# A real MultimodalRAG.ask() call will now emit spans — e.g.:
#   rag = MultimodalRAG(retriever=retriever, vlm=vlm)
#   result = rag.ask("What is attention?")
#
# You will see spans like:
#   mrta.multimodal_retriever.retrieve
#       retrieval.text_candidates = 5
#       latency.text_retrieval = 0.023
#   mrta.multimodal_rag.ask
#       generation.text_evidence_count = 3
#       latency.vlm = 4.2

print("Tracer configured. Run rag.ask(...) to see spans.")

## 9.9 Docker / Reproducibility Notes

### Text-only mode (default)

```bash
docker compose up --build
```

No additional setup needed. The API starts in text-only mode. Ollama runs on
the host at `http://host.docker.internal:11434`.

### Multimodal mode (optional)

Two additional components are required:

**1. CLIP model** — downloads automatically from HuggingFace on first use:
```bash
# No manual step needed — open_clip downloads ViT-B-32 (~350 MB) automatically.
# Set CLIP_MODEL in .env to change the model.
```

**2. Vision model** — must be pulled into Ollama manually:
```bash
ollama pull qwen2.5vl:7b
```

Then restart the API:
```bash
docker compose restart api
```

The API logs `"Multimodal stack initialised"` on startup when CLIP + VLM are
available, or `"Multimodal stack unavailable"` when they are not. Text RAG
continues to work in either case.

### Reproducibility checklist

| Component | How to pin |
|---|---|
| Python deps | `requirements.txt` (pip-compile) |
| CLIP model | `CLIP_MODEL=ViT-B-32` in `.env` |
| Vision model | `OLLAMA_VLM_MODEL=qwen2.5vl:7b` in `.env` |
| Text LLM | `OLLAMA_LLM_MODEL=llama3.2:latest` in `.env` |
| Embedding model | `EMBEDDING_MODEL=nomic-embed-text` in `.env` |
| Eval dataset | `tests/evaluation/multimodal_benchmark.json` (version-controlled) |